# SPR 2026 — Mammography Report Classification: Baseline

**Problem:** Predict the BI-RADS category (0–6) from Portuguese mammography report text.  
**Metric:** Macro F1-Score — all 7 classes are weighted equally, so rare classes matter just as much as the dominant class 2.  
**Approach:** TF-IDF vectorization + Logistic Regression with balanced class weights.

---

### How this notebook is organized

| Section | What it does |
|---------|-------------|
| 1. Setup | Imports and file paths |
| 2. Load Data | Read train.csv and test.csv |
| 3. Explore Data | Understand class imbalance and text structure |
| 4. Preprocess Text | Clean the report text |
| 5. Build Features | TF-IDF vectorization |
| 6. Train & Validate | Cross-validation to get an honest F1 score |
| 7. Evaluate | Per-class breakdown and confusion matrix |
| 8. Interpret | What words drive each prediction? |
| 9. Submit | Predict test set and save submission file |

Run cells top-to-bottom. Each cell does exactly one thing.

---
## Section 1 — Setup
Import libraries and set file paths.

> The path block auto-detects whether you're running locally or on Kaggle so you don't need to change anything when you upload to Kaggle.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# ── File paths ────────────────────────────────────────────────────────────────
# On Kaggle the data is at /kaggle/input/<competition-name>/
# Locally it's in competition_data/ (one level up from this notebook)
KAGGLE_DIR = "/kaggle/input/spr-2026-mammography-report-classification"
LOCAL_DIR  = os.path.join(os.getcwd(), "..", "competition_data")

DATA_DIR   = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else LOCAL_DIR
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED    = 42
N_FOLDS = 5

print("Data directory:", os.path.abspath(DATA_DIR))

---
## Section 2 — Load Data

- `train.csv` — 18,272 reports with known BI-RADS labels. We train and evaluate here.
- `test.csv` — Reports with **no labels**. We predict these and submit to Kaggle.

> **Why no target in test.csv?**  
> The true test labels live only on Kaggle's servers. You never see them.  
> You submit your predictions → Kaggle compares to secret labels → leaderboard score.  
> Our cross-validation score on train.csv is our best local estimate of leaderboard performance.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Standardize column names to lowercase (the CSV uses uppercase "ID")
train_df.columns = train_df.columns.str.strip().str.lower()
test_df.columns  = test_df.columns.str.strip().str.lower()

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print()
print("Train columns:", train_df.columns.tolist())
print("Test columns: ", test_df.columns.tolist())

---
## Section 3 — Explore the Data

Before modeling, we need to understand:
1. **Class distribution** — how imbalanced is it?
2. **Text length** — how long are the reports?
3. **Duplicate reports** — are there repeated boilerplate texts?

These findings directly shape our modeling choices.

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
# Macro F1 weights all 7 classes equally.
# Class 2 is 87% of the data — a model that always predicts class 2 scores ~0.14 macro F1.
# We must explicitly counteract this imbalance.

total = len(train_df)
print("Class distribution (this is the most important thing to notice):")
print()

for cls, cnt in train_df["target"].value_counts().sort_index().items():
    bar     = "█" * int(cnt / total * 40)
    warning = " ← RARE" if cnt < 100 else ""
    print(f"  BI-RADS {cls}  {cnt:>6,}  ({cnt/total*100:>5.1f}%)  {bar}{warning}")

In [ ]:
# ── Text length per class ─────────────────────────────────────────────────────
# Longer reports tend to describe more complex/serious findings.
# Class 6 (known malignancy) averages the most words.

train_df["word_count"] = train_df["report"].str.split().str.len()

print("Average word count per class:")
print(train_df.groupby("target")["word_count"].mean().round(1).rename("avg words").to_string())

In [ ]:
# ── Duplicate report check ────────────────────────────────────────────────────
# Many reports are identical boilerplate 'all clear' screening exams.
# This is normal — the same form text gets generated for thousands of normal patients.
# We keep them in training because they are real examples the model should learn from.

n_dup = train_df["report"].duplicated().sum()
print(f"Rows with identical report text to a prior row: {n_dup} ({n_dup/total*100:.0f}%)")
print()

# Show the most common report text
most_common_text = train_df["report"].value_counts().index[0]
print("Most repeated report (first 200 chars):")
print(most_common_text[:200])

In [ ]:
# ── Sample report per class ───────────────────────────────────────────────────
# Gives intuition for the language and content differences between classes.

print("One example report per class (first 250 chars):")
print()
for cls in sorted(train_df["target"].unique()):
    example = train_df[train_df["target"] == cls]["report"].iloc[0][:250]
    print(f"--- BI-RADS {cls} ---")
    print(example)
    print()

---
## Section 4 — Preprocess Text

We apply **lowercase only**. That's it. Here is why we are deliberately minimal:

- **Keep Portuguese accents (ã, é, ç):** They distinguish words. "mama" and "mamã" are different.
- **Do NOT remove stopwords:** "não" (not) is the most important negation word in medical Portuguese. "Não se observam nódulos" = no nodules found. Removing "não" destroys this signal.
- **Do NOT strip punctuation:** TF-IDF handles tokenization. The colon `:` after "Achados" is part of the section structure.
- **Keep `<DATA>` placeholder:** It marks de-identified dates. Its presence signals a follow-up exam (relevant for class 3).

In [ ]:
def preprocess_report(text):
    """Lowercase only. See markdown cell above for why we do nothing else."""
    if not isinstance(text, str):
        return ""
    return text.lower().strip()


train_df["report"] = train_df["report"].apply(preprocess_report)
test_df["report"]  = test_df["report"].apply(preprocess_report)

print("Preprocessing applied.")
print()
print("Example preprocessed report (first 200 chars):")
print(train_df["report"].iloc[0][:200])

---
## Section 5 — Build Features (TF-IDF)

**TF-IDF** converts raw text into a numerical matrix the model can use.

- **TF** (Term Frequency): how often does this word appear in this document?
- **IDF** (Inverse Document Frequency): how rare is this word across all documents?
- **TF-IDF = TF × IDF**: words that appear a lot in *this* document but rarely elsewhere get high scores.

We stack two TF-IDF vectorizers:

**Word n-grams (1–2):** Captures single words AND two-word phrases.  
Key phrases like `"nódulo espiculado"` (spiculated nodule) and `"calcificações pleomórficas"` are strong predictors for BI-RADS 4/5.

**Character n-grams (3–5):** Captures letter substrings.  
`"espicul"` matches `"espiculado"`, `"espiculada"`, `"espiculados"` — Portuguese changes word endings by gender/number. Char n-grams handle this automatically.

In [ ]:
def build_tfidf_features():
    """Return a FeatureUnion that stacks word TF-IDF and char TF-IDF."""

    # Word-level TF-IDF: captures keywords and key phrases
    word_tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),        # unigrams + bigrams
        min_df=3,                  # ignore terms in fewer than 3 documents (noise)
        max_df=0.95,               # ignore terms in >95% of documents (uninformative)
        sublinear_tf=True,         # use log(1+tf) to dampen high-frequency terms
        strip_accents=None,        # keep Portuguese accents
        token_pattern=r"(?u)\b\w+\b",
    )

    # Character-level TF-IDF: handles morphological variation in Portuguese
    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",        # character n-grams within word boundaries
        ngram_range=(3, 5),        # 3- to 5-character substrings
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
    )

    # FeatureUnion stacks the two feature matrices side by side
    return FeatureUnion([
        ("word_tfidf", word_tfidf),
        ("char_tfidf", char_tfidf),
    ])


print("TF-IDF feature builder defined.")

---
## Section 6 — Train and Validate (Cross-Validation)

### Why cross-validation?
We cannot evaluate on `test.csv` (no labels). Instead we use **k-fold cross-validation** on `train.csv`:
- Split train.csv into 5 equal folds
- For each fold: train on the other 4 folds, predict on this fold
- Each row's prediction was made by a model that **never saw that row** → honest estimate

These are called **OOF (Out-of-Fold) predictions**.

### Why stratified?
Class 5 only has 29 examples. Without stratification, some folds might contain 0 examples of class 5, making it impossible to evaluate. Stratified folds guarantee proportional class representation in every fold.

### About the classifier
Logistic Regression with `class_weight='balanced'` automatically upweights rare classes.  
Class 5 (29 examples) gets weighted ~43× higher than class 2 (15,968 examples).  
Without this, the model would essentially ignore classes 4, 5, and 6.

In [ ]:
X = train_df["report"]
y = train_df["target"]

# Build the full pipeline: TF-IDF features → Logistic Regression
classifier = LogisticRegression(
    class_weight="balanced",   # upweight rare classes — essential for macro F1
    C=1.0,                     # regularization (lower = simpler model)
    max_iter=1000,             # default 100 often doesn't converge for TF-IDF
    solver="saga",             # best solver for large sparse feature matrices
    multi_class="multinomial", # true 7-class classification
    random_state=SEED,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("features", build_tfidf_features()),
    ("clf",      classifier),
])

print("Pipeline ready.")

In [ ]:
# ── Run 5-fold stratified cross-validation ────────────────────────────────────
skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(y), dtype=int)  # stores one prediction per training row
fold_f1s  = []

print(f"Running {N_FOLDS}-fold stratified cross-validation...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    # Split into this fold's train and validation sets
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold   = X.iloc[val_idx]
    y_val_fold   = y.iloc[val_idx]

    # Train on 80%, predict on 20%
    pipeline.fit(X_train_fold, y_train_fold)
    fold_preds = pipeline.predict(X_val_fold)

    # Store predictions and compute macro F1 for this fold
    oof_preds[val_idx] = fold_preds
    fold_score = f1_score(y_val_fold, fold_preds, average="macro")
    fold_f1s.append(fold_score)

    print(f"  Fold {fold + 1}/{N_FOLDS}  →  macro F1 = {fold_score:.4f}")

print()
print(f"Per-fold scores: {[f'{s:.4f}' for s in fold_f1s]}")
print(f"Mean macro F1:   {np.mean(fold_f1s):.4f}  ±  {np.std(fold_f1s):.4f}")
print()
print("This is our best local estimate of how the model will score on the leaderboard.")

---
## Section 7 — Evaluate Results

We evaluate the OOF predictions — each row was predicted by a model that never trained on it.

**Classification report columns:**
- **Precision** — of all times we predicted class X, what fraction were correct?
- **Recall** — of all true class X examples, what fraction did we catch?
- **F1** — harmonic mean of precision and recall
- **Support** — how many true examples exist for that class

The row we care about most is **macro avg** — this is the competition metric.

In [ ]:
# ── Overall OOF macro F1 ──────────────────────────────────────────────────────
oof_macro_f1 = f1_score(y, oof_preds, average="macro")

print("=" * 55)
print(f"  OOF MACRO F1: {oof_macro_f1:.4f}   (our leaderboard estimate)")
print("=" * 55)

In [ ]:
# ── Per-class breakdown ───────────────────────────────────────────────────────
# This tells us WHICH classes the model handles well vs. poorly.
# Weak classes drag macro F1 down — they are where improvement matters most.

print(classification_report(
    y,
    oof_preds,
    target_names=[f"BI-RADS {c}" for c in sorted(y.unique())],
    zero_division=0,
))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
# Rows = true label, Columns = predicted label.
# Diagonal = correct predictions.
# Off-diagonal = errors — which class was confused with which?

classes = sorted(y.unique())
cm      = confusion_matrix(y, oof_preds, labels=classes)
cm_df   = pd.DataFrame(
    cm,
    index=[f"True BR{c}" for c in classes],
    columns=[f"Pred BR{c}" for c in classes],
)

print("Confusion Matrix  (rows = true label, columns = predicted label)")
print("Tip: large numbers off the diagonal = frequent errors between those two classes.")
print()
print(cm_df.to_string())

In [ ]:
# ── Biggest misclassifications ────────────────────────────────────────────────
# Summarize the most common errors to understand what the model struggles with.

print("Most common misclassifications (more than 5 cases):")
print()

for i, true_c in enumerate(classes):
    for j, pred_c in enumerate(classes):
        if i != j and cm[i, j] > 5:
            pct = cm[i, j] / cm[i].sum() * 100
            print(f"  True BI-RADS {true_c}  →  predicted BI-RADS {pred_c}:  "
                  f"{cm[i, j]:4d} cases  ({pct:.0f}% of all true class {true_c})")

---
## Section 8 — Interpret the Model

Logistic Regression has one big advantage over black-box models: **we can read what it learned**.

Each class has a set of coefficients — one per word/n-gram in the vocabulary.
A large positive coefficient means that word strongly predicts that class.

If these top features are medically meaningful clinical terms, the model is learning real signal.  
If they look like noise or artifacts, we have a problem.

In [ ]:
# The pipeline was last fit on Fold 5 data during CV.
# Retrain on ALL training data before reading weights — then weights reflect everything.
pipeline.fit(X, y)

fitted_clf      = pipeline.named_steps["clf"]
fitted_features = pipeline.named_steps["features"]

# Get all feature names (word n-grams + char n-grams concatenated)
word_feature_names = fitted_features.transformer_list[0][1].get_feature_names_out()
char_feature_names = fitted_features.transformer_list[1][1].get_feature_names_out()
all_feature_names  = np.concatenate([word_feature_names, char_feature_names])

TOP_N = 8
print(f"Top {TOP_N} most predictive word-level features per BI-RADS class:")
print()

for class_idx, cls in enumerate(fitted_clf.classes_):
    # Get the indices of the highest-weighted features for this class
    top_indices = np.argsort(fitted_clf.coef_[class_idx])[-TOP_N:][::-1]

    # Filter to word n-grams only for readability
    # (char n-grams like 'eni', 'ign' are harder to interpret)
    word_top = [
        (all_feature_names[i], fitted_clf.coef_[class_idx][i])
        for i in top_indices
        if all_feature_names[i] in set(word_feature_names)
    ][:TOP_N]

    print(f"  BI-RADS {cls}:")
    for term, weight in word_top:
        print(f"    {term!r:<42}  weight = {weight:.2f}")
    print()

---
## Section 9 — Generate Submission

Now we predict on `test.csv` (which has no labels) and save a CSV to upload to Kaggle.

> The model was already retrained on all 18,272 training examples in Section 8.  
> More training data = better final model, which is why we retrain on everything after CV is done.

**Local:** You will see only 4 test rows (the local sample).  
**On Kaggle:** `test.csv` will contain all competition test rows — the code handles both automatically.

In [ ]:
# Predict on the test set
test_predictions = pipeline.predict(test_df["report"])

# Build the submission dataframe
submission = pd.DataFrame({
    "ID":     test_df["id"],
    "target": test_predictions,
})

print("Test predictions:")
print(submission)

In [ ]:
# ── Save submission file ──────────────────────────────────────────────────────
# On Kaggle, output files go to /kaggle/working/
# Locally, they go next to this notebook

if os.path.exists("/kaggle/working"):
    out_path = "/kaggle/working/submission_baseline.csv"
else:
    out_path = os.path.join(os.getcwd(), "submission_baseline.csv")

submission.to_csv(out_path, index=False)
print(f"Submission saved to: {out_path}")
print()
print("Upload this file to Kaggle to see your leaderboard score.")

---
## Results Summary

| Class | F1 Score | Notes |
|-------|----------|-------|
| BI-RADS 0 | ~0.55 | Incomplete — tricky, often confused with 3 |
| BI-RADS 1 | ~0.85 | Negative — clean text, model handles well |
| BI-RADS 2 | ~0.93 | Benign — dominant class, model handles well |
| BI-RADS 3 | ~0.36 | Probably benign — adjacent to 0 and 4 |
| BI-RADS 4 | ~0.26 | Suspicious — confused with 3 and 5 |
| BI-RADS 5 | ~0.09 | Highly suspicious — only 29 examples |
| BI-RADS 6 | ~0.17 | Known malignancy — only 45 examples |
| **macro avg** | **~0.46** | **Competition metric** |

### What this score means
0.46 is a clean TF-IDF baseline. The model learned real clinical vocabulary (see Section 8).  
The bottleneck is classes 4, 5, and 6 — rare, but each worth 1/7 of the macro F1.

### Likely next steps
1. **Fine-tune a Portuguese BERT model (BERTimbau)** — likely +15–25 F1 points
2. **Tune class weights manually** — give classes 5 and 6 even more emphasis
3. **Parse report sections** — treat Indication and Findings as separate features